# DelDel: tour visual de fronteras, superficies y regiones

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jcval94/PureSheShe/blob/main/notebooks/deldel_visual_tour.ipynb)

Este notebook recorre las capacidades principales de **DelDel** sobre un problema sintético 4D: generación de datos, extracción de cambios de clase, ajuste de fronteras lineales y cuadráticas, búsqueda de reglas de baja dimensión y visualización interactiva de regiones.

Está pensado para ejecutarse de arriba abajo en CPU (aprox. 15–30 s; la instalación inicial puede tardar algo más). Los gráficos de Plotly se pueden rotar, ampliar y filtrar desde la leyenda.

## 1. Preparar el entorno

En Colab, la celda clona el repositorio e instala DelDel en modo editable. En un checkout local reutiliza el directorio actual y añade el layout `src/` a la sesión sin modificar el entorno.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/jcval94/PureSheShe.git"
repo = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").exists() and (path / "src/deldel").exists()),
    None,
)

if repo is None:
    repo = Path("/content/PureSheShe") if Path("/content").exists() else Path.cwd() / "PureSheShe"
    if not repo.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo)])

os.chdir(repo)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-build-isolation", "-e", ".", "matplotlib>=3.7",
    ])
else:
    # En un checkout local basta exponer el layout src/; no modifica tu entorno.
    sys.path[:0] = [str(repo / "src"), str(repo)]
print(f"DelDel listo desde: {Path.cwd()}")

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from deldel import (
    ChangePointConfig,
    DelDel,
    DelDelConfig,
    compute_frontier_planes_all_modes,
    compute_frontier_planes_weighted,
    describe_regions_metrics,
    describe_regions_report,
    find_low_dim_spaces,
    fit_quadrics_from_records_weighted,
    plot_frontiers_implicit_interactive_v2,
    plot_planes_with_point_lines,
    plot_selected_regions_interactive,
    prune_and_orient_planes_unified_globalmaj,
)
from deldel.datasets import make_corner_class_dataset, plot_corner_class_dataset

pio.renderers.default = "colab" if IN_COLAB else "notebook_connected"
SEED = 42

## 2. Ver el dataset canónico de DelDel

La clase 1 ocupa seis esquinas de un hipercubo 4D; las clases 0 y 2 forman regiones compactas. La librería incluye una proyección PCA 3D y una matriz de dispersión para inspeccionar ambas estructuras.

In [ ]:
X, y, feature_names = make_corner_class_dataset(
    n_per_cluster=45,
    std_class1=0.4,
    std_other=0.7,
    a=3.0,
    random_state=SEED,
)

print(f"X: {X.shape} | clases: {dict(zip(*np.unique(y, return_counts=True)))}")
dataset_figures = plot_corner_class_dataset(X, y, feature_names)

## 3. Entrenar un modelo y extraer cambios de decisión

DelDel instrumenta un clasificador ya entrenado. Muestrea segmentos entre observaciones y conserva los que revelan cambios de clase; esos `records_` alimentan el resto del análisis.

In [ ]:
model = RandomForestClassifier(
    n_estimators=100, random_state=SEED, n_jobs=-1
).fit(X, y)

engine = DelDel(
    DelDelConfig(segments_target=180, random_state=SEED),
    ChangePointConfig(enabled=False),
).fit(X, model)
records = engine.records_

print(f"Accuracy de entrenamiento: {model.score(X, y):.3f}")
print(f"Registros de cambio extraídos: {len(records)}")
ConfusionMatrixDisplay.from_estimator(model, X, y, cmap="Blues");

## 4. Fronteras ponderadas interactivas

Aquí aparecen juntas las muestras, los puntos frontera, los segmentos que condujeron a ellas y un plano ponderado por cada par de clases detectado. La vista es un corte sobre `x1`, `x2` y `x3`; `dims` permite elegir cualquier combinación 1D, 2D o 3D.

In [ ]:
weighted_planes = compute_frontier_planes_weighted(
    records, weight_map="softmax"
)

fig_frontiers = plot_frontiers_implicit_interactive_v2(
    records, X, y,
    planes=weighted_planes,
    dims=(0, 1, 2),
    feature_names=feature_names,
    detail="low",
    grid_res_3d=24,
    show_directions="lines",
    arrows_per_pair=15,
    plane_opacity=0.14,
    show=False,
    return_fig=True,
    title="DelDel — puntos frontera, trayectorias y planos ponderados",
)
fig_frontiers.show()

## 5. Superficies no lineales

Cuando un plano no basta, DelDel ajusta cuádricas ponderadas a los mismos registros. Las isosuperficies siguientes dejan ver la curvatura de las fronteras entre clases.

In [ ]:
quadrics = fit_quadrics_from_records_weighted(
    records, mode="logistic", C=6.0, density_k=6
)

fig_quadrics = plot_frontiers_implicit_interactive_v2(
    records, X, y,
    quadrics=quadrics,
    dims=(0, 1, 2),
    feature_names=feature_names,
    detail="low",
    grid_res_3d=24,
    show_planes=False,
    show_directions="none",
    quadric_alpha=0.30,
    show=False,
    return_fig=True,
    title="DelDel — fronteras cuadráticas ponderadas",
)
fig_quadrics.show()

## 6. Familias de planos por par de clases

El modo `C` busca varios planos para representar fronteras multimodales. Este gráfico relaciona cada plano con los puntos y segmentos que lo sustentan.

In [ ]:
frontier_families = compute_frontier_planes_all_modes(
    records,
    mode="C",
    min_cluster_size=5,
    max_models_per_round=3,
    seed=SEED,
)

fig_families = plot_planes_with_point_lines(
    frontier_families,
    records=records,
    X=X, y=y,
    dims=(0, 1, 2),
    feature_names=feature_names,
    show_cloud=True,
    max_points=80,
    show=False,
    return_fig=True,
    title="DelDel — familias de planos y evidencia local",
)
fig_families.show()

## 7. Convertir fronteras en reglas interpretables

La poda global orienta los semiespacios hacia sus clases. Después, `find_low_dim_spaces` combina planos y devuelve reglas 1D–3D con soporte, precisión, recall, F1 y lift.

In [ ]:
selection = prune_and_orient_planes_unified_globalmaj(
    frontier_families, X, y,
    feature_names=feature_names,
    max_k=6,
    min_region_size=12,
    min_abs_diff=0.01,
    min_rel_lift=0.02,
)

valuable = find_low_dim_spaces(
    X, y, selection,
    feature_names=feature_names,
    max_planes_in_rule=3,
    max_planes_per_pair=4,
    min_support=15,
    min_rel_gain_f1=0.01,
    min_lift_prec=1.05,
    consider_dims_up_to=3,
    rng_seed=SEED,
)

print(describe_regions_report(valuable, top_per_class=2, dataset_size=len(X)))

### Calidad y complejidad de las reglas

La salida estructurada de `describe_regions_metrics` se puede llevar directamente a pandas/Plotly. El tamaño representa el soporte y el color identifica la clase objetivo.

In [ ]:
metrics_df = pd.DataFrame(
    describe_regions_metrics(valuable, top_per_class=8, dataset_size=len(X))
).drop_duplicates(subset=["class_id", "region_id"])
metrics_df["dims_count"] = metrics_df["dims"].map(len)
display(metrics_df[["class_id", "region_id", "dims", "f1", "lift_precision", "support"]].head(10))

fig_metrics = px.scatter(
    metrics_df,
    x="lift_precision", y="f1",
    color=metrics_df["class_id"].astype(str),
    size="support", symbol="dims_count",
    hover_data=["region_id", "dims", "precision", "recall"],
    labels={"color": "clase", "dims_count": "dimensiones"},
    title="Reglas encontradas: F1 vs. lift de precisión",
)
fig_metrics.show()

## 8. Inspeccionar la mejor región en su propio subespacio

La última vista elige la mejor regla de al menos dos dimensiones y dibuja la intersección de sus semiespacios. Cambia `best_region` por cualquier fila del DataFrame anterior para explorar otra regla.

In [ ]:
region_candidates = [
    region
    for regions in valuable.values()
    for region in regions
    if len(region.get("dims", ())) >= 2
]
best_region = max(region_candidates, key=lambda region: region["metrics"]["f1"])
region_dims = tuple(best_region["dims"][:3])

print(describe_regions_report(
    valuable, region_id=best_region["region_id"], dataset_size=len(X)
))

fig_region = plot_selected_regions_interactive(
    selection, X, y,
    selected_region_ids=[best_region["region_id"]],
    valuable=valuable,
    dims=region_dims,
    feature_names=feature_names,
    volume_res_3d=24,
    show=False,
    return_fig=True,
    title=f"Mejor región {len(region_dims)}D — {best_region['region_id']}",
)
fig_region.show()

## Siguientes experimentos

- Cambia `dims=(0, 1, 2)` para explorar otros cortes del espacio 4D.
- Activa `ChangePointConfig(enabled=True, mode="treefast")` para refinar los puntos de cambio.
- Sube `segments_target`, `grid_res_3d` o `max_models_per_round` para obtener más detalle.
- Prueba `find_comb_dim_spaces_full` si necesitas reglas AND, OR o DNF.
- Usa `make_high_dim_classification_dataset` y `MultiClassSubspaceExplorer` para análisis de muchas variables.